# Bayesian Optimization for Hyperparameter Tuning

Grid and random search are **memoryless**: each new hyperparameter combination is chosen without looking at how the previous ones scored. When a single evaluation is expensive (a full cross-validated model fit), that waste hurts.

**Bayesian optimization (BO)** is *sequential model-based optimization*. It keeps a cheap probabilistic **surrogate** of the objective (score as a function of hyperparameters), and uses an **acquisition function** to decide where to look next — balancing *exploiting* regions that already score well against *exploring* regions it is uncertain about.

In this notebook we:

1. Define an expensive objective — cross-validated accuracy of a `RandomForestClassifier` on the breast-cancer dataset — over a 4-dimensional search space.
2. Run two **baselines**: an exhaustive `GridSearchCV` and a `Random search`, recording their best score and their budget (number of model-fits).
3. Run **Bayesian optimization**: try `optuna` (TPE sampler); if it is not installed, fall back to a hand-rolled **Gaussian-Process + Expected-Improvement** loop that implements the same core idea with scikit-learn only.
4. Plot **best-score-so-far vs number of trials** to show BO reaching a good score in fewer evaluations.

> This notebook runs fully **offline** with `numpy, pandas, scikit-learn, scipy, matplotlib, seaborn`. `optuna` is optional; the GP fallback guarantees the core idea always runs.

In [ ]:
# ---- Core numerical + data stack (all offline, all pre-installed) ----
import numpy as np                     # arrays / vectorized math
import pandas as pd                    # tidy result tables
import matplotlib.pyplot as plt        # plotting
import seaborn as sns                  # nicer default styling for the plots
from scipy.stats import norm           # Gaussian CDF/PDF -> used by Expected Improvement

# ---- scikit-learn: the model we tune + the tuning utilities ----
from sklearn.datasets import load_breast_cancer                      # small offline dataset
from sklearn.ensemble import RandomForestClassifier                  # estimator we tune
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.gaussian_process import GaussianProcessRegressor        # the SURROGATE model
from sklearn.gaussian_process.kernels import Matern, ConstantKernel, WhiteKernel

sns.set_theme(style='whitegrid')       # apply seaborn styling globally
rng_global = np.random.default_rng(42) # one Generator we thread through the notebook
np.random.seed(42)                     # legacy global seed, for anything that uses it

print('imports OK')

## Installing optuna (optional)

`optuna` is a popular BO library but is **not installed** in this environment (and cannot be installed offline). If you want the TPE backend, install it in a networked environment with:

```
pip install optuna
```

The code below is written so that if the import fails it automatically falls back to a scikit-learn Gaussian-Process surrogate — so **the notebook always runs end-to-end**.

## 1. The tuning target (an expensive objective)

We tune a `RandomForestClassifier` on the **breast-cancer** dataset (569 samples, 30 features, binary target). The objective is the **mean 3-fold cross-validated accuracy**:

$$f(\theta) = \frac{1}{K}\sum_{k=1}^{K} \text{acc}_k(\theta), \qquad \theta = (\text{n\_estimators}, \text{max\_depth}, \text{min\_samples\_split}, \text{max\_features})$$

Every tuner tries to **maximize** $f(\theta)$. One call to `objective()` costs $K=3$ model fits, so evaluations are the currency we want to spend sparingly. Internally we work in a normalized unit cube $[0,1]^d$ and `decode` back to real hyperparameters — this keeps the surrogate model well-scaled across dimensions of very different magnitude.

In [ ]:
# Load a small, clean, fully-offline dataset: 569 samples, 30 features, binary target.
data = load_breast_cancer()
X, y = data.data, data.target
print(f'dataset: X={X.shape}, class counts={np.bincount(y)}')

# 3-fold stratified CV. A FIXED random_state makes the objective DETERMINISTIC given params,
# which keeps the grid / random / Bayesian comparison fair and reproducible.
CV = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)

# ------------------------------------------------------------------
# The hyperparameter SEARCH SPACE.
# Each entry maps: name -> (type, low, high).  Three integers + one float.
# ------------------------------------------------------------------
SPACE = {
    'n_estimators':      ('int',   20,  80),   # number of trees in the forest
    'max_depth':         ('int',    2,  20),   # max depth of each tree
    'min_samples_split': ('int',    2,  20),   # min samples required to split a node
    'max_features':      ('float', 0.1, 1.0),  # fraction of features considered per split
}
DIM_NAMES = list(SPACE.keys())   # a STABLE ordering of the dimensions
N_DIMS = len(DIM_NAMES)

# A global tally of how many CV-evaluations each strategy costs (its 'budget').
EVAL_COUNT = 0

def objective(params):
    """Mean 3-fold CV accuracy for a RandomForest with the given hyperparameters.
    This is the expensive black box every tuner is trying to MAXIMIZE."""
    global EVAL_COUNT
    EVAL_COUNT += 1                                  # one call == one CV evaluation (= K fits)
    model = RandomForestClassifier(
        random_state=0, n_jobs=1, **params          # fixed seed -> deterministic score
    )
    scores = cross_val_score(model, X, y, cv=CV, scoring='accuracy')
    return float(scores.mean())

# ---- helpers to move between a unit vector u in [0,1]^d and a real params dict ----
def decode(u):
    """Map a point u in the unit hypercube [0,1]^d to a concrete hyperparameter dict."""
    params = {}
    for value, name in zip(u, DIM_NAMES):
        kind, low, high = SPACE[name]
        if kind == 'int':
            params[name] = int(round(low + value * (high - low)))  # scale then round to int
        else:
            params[name] = float(low + value * (high - low))       # scale as a float
    return params

def sample_unit(n, generator):
    """Draw n points uniformly from the unit hypercube [0,1]^d."""
    return generator.random((n, N_DIMS))

# quick smoke test: evaluate one point in the middle of the space
_demo = decode(np.full(N_DIMS, 0.5))
print('sample params:', _demo, '-> cv acc:', round(objective(_demo), 4))
EVAL_COUNT = 0   # reset the counter after the smoke test

## 2. Baselines: Grid search and Random search

**Grid search** evaluates every point on a predefined mesh — its cost grows *exponentially* with the number of hyperparameters (the curse of dimensionality). **Random search** samples points independently at random; it often beats grid search per-evaluation because it spreads samples across every dimension instead of wasting fits on a coarse lattice. Neither uses information from past trials — both are **memoryless**.

We give random search a fixed budget of `N_TRIALS` evaluations — the **same budget** Bayesian optimization will get, so the comparison is fair.

In [ ]:
# ==========================================================================
# BASELINE 1 - GridSearchCV: exhaustively try every combination on a coarse grid.
# ==========================================================================
param_grid = {
    'n_estimators':      [30, 80],
    'max_depth':         [6, 18],
    'min_samples_split': [2],
    'max_features':      [0.3, 0.7],
}
n_combos = int(np.prod([len(v) for v in param_grid.values()]))   # size of the lattice
grid = GridSearchCV(
    RandomForestClassifier(random_state=0, n_jobs=1),
    param_grid, cv=CV, scoring='accuracy', n_jobs=1,
)
grid.fit(X, y)
# budget = combinations * folds (each combination is fit once per fold)
grid_fits = n_combos * CV.get_n_splits()
print(f'GridSearchCV : {n_combos} combos x {CV.get_n_splits()} folds = {grid_fits} model-fits')
print(f'GridSearchCV : best acc    = {grid.best_score_:.4f}')
print(f'GridSearchCV : best params = {grid.best_params_}')

# ==========================================================================
# BASELINE 2 - Random search: sample N_TRIALS random points from the SAME space.
# This gives a fair, trial-by-trial curve to compare against Bayesian search.
# ==========================================================================
N_TRIALS = 12                                   # shared budget for random & Bayesian
rng_rand = np.random.default_rng(7)             # independent RNG for random search
EVAL_COUNT = 0

rand_scores = []                                # score obtained at each trial
for u in sample_unit(N_TRIALS, rng_rand):       # one random unit-cube point per trial
    rand_scores.append(objective(decode(u)))    # evaluate the expensive black box
rand_scores = np.array(rand_scores)

# 'best score so far' after each trial -> the efficiency curve we will plot
rand_best_curve = np.maximum.accumulate(rand_scores)
print(f'\nRandom search: {EVAL_COUNT} CV-evals = {EVAL_COUNT * CV.get_n_splits()} model-fits')
print(f'Random search: best acc = {rand_scores.max():.4f}')

## 3. Bayesian optimization

BO repeats three steps until the budget runs out:

1. **Surrogate.** Fit a cheap probabilistic model of $f(\theta)$ from the trials seen so far. A **Gaussian Process (GP)** is the classic choice: it predicts a mean $\mu(\theta)$ *and* an uncertainty $\sigma(\theta)$ at every point. (`optuna`'s TPE instead models $p(\theta\mid\text{good})$ vs $p(\theta\mid\text{bad})$ — a different surrogate, same spirit.)
2. **Acquisition.** Turn $(\mu, \sigma)$ into a score that says *where to sample next*. We use **Expected Improvement (EI)**:

$$\text{EI}(\theta) = \mathbb{E}\big[\max(f(\theta) - f^{+}, 0)\big] = (\mu - f^{+} - \xi)\,\Phi(z) + \sigma\,\phi(z), \qquad z = \frac{\mu - f^{+} - \xi}{\sigma}$$

   where $f^{+}$ is the best score so far, $\Phi, \phi$ are the standard-normal CDF/PDF, and $\xi$ nudges exploration. EI is large when the predicted mean beats the incumbent (**exploitation**) **or** when uncertainty is high (**exploration**).
3. **Evaluate** the real (expensive) objective at the acquisition maximizer, add it to the data, and repeat.

Because each new trial is *informed* by all previous ones, BO concentrates its budget where good scores are likely — the key advantage when evaluations are expensive.

In [ ]:
# ==========================================================================
# BAYESIAN OPTIMIZATION
# First TRY optuna (TPE sampler). If optuna is not installed (it is not here),
# fall back to a hand-rolled Gaussian-Process + Expected-Improvement loop.
# Either way we spend the SAME budget (N_TRIALS) for a fair comparison.
# ==========================================================================
bayes_scores = None          # per-trial scores in the order they were evaluated
bayes_best_params = None     # best hyperparameters found
bayes_backend = None         # which backend actually ran

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)   # keep the output quiet

    def optuna_objective(trial):
        # optuna's own way of declaring the SAME search space
        params = {
            'n_estimators':      trial.suggest_int('n_estimators', 20, 80),
            'max_depth':         trial.suggest_int('max_depth', 2, 20),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'max_features':      trial.suggest_float('max_features', 0.1, 1.0),
        }
        return objective(params)

    sampler = optuna.samplers.TPESampler(seed=42)          # Tree-structured Parzen Estimator
    study = optuna.create_study(direction='maximize', sampler=sampler)
    study.optimize(optuna_objective, n_trials=N_TRIALS)    # same budget as random search

    bayes_scores = np.array([t.value for t in study.trials])
    bayes_best_params = study.best_params
    bayes_backend = 'optuna (TPE)'

except ImportError:
    # ---------- FALLBACK: GP surrogate + Expected Improvement acquisition ----------
    print('optuna not available -> running the hand-rolled GP + Expected-Improvement loop')

    def expected_improvement(mu, sigma, best_f, xi=0.01):
        """EI for MAXIMIZATION. Rewards candidates whose predicted score (mu) beats the
        current best AND candidates the surrogate is very UNSURE about (large sigma)."""
        sigma = np.maximum(sigma, 1e-9)             # guard against divide-by-zero
        improve = mu - best_f - xi                  # how much we expect to gain over the best
        z = improve / sigma
        return improve * norm.cdf(z) + sigma * norm.pdf(z)   # exploit term + explore term

    # Matern kernel = a smooth-but-not-too-smooth surrogate; WhiteKernel absorbs CV noise.
    kernel = ConstantKernel(1.0) * Matern(length_scale=0.3, nu=2.5) + WhiteKernel(1e-3)

    rng_bo = np.random.default_rng(7)               # same seed family as random search
    N_INIT = 5                                      # random warm-up points before the GP kicks in

    # --- 1. warm-up: a few random evaluations so the GP has something to fit ---
    X_obs = sample_unit(N_INIT, rng_bo)             # observed points in [0,1]^d
    y_obs = np.array([objective(decode(u)) for u in X_obs])

    # --- 2. sequential model-based loop for the remaining budget ---
    for _ in range(N_TRIALS - N_INIT):
        gp = GaussianProcessRegressor(
            kernel=kernel, normalize_y=True, n_restarts_optimizer=1, random_state=0
        )
        gp.fit(X_obs, y_obs)                        # SURROGATE: a cheap stand-in for the objective

        # propose a big pool of candidate points, score them all with EI, keep the best
        cand = sample_unit(300, rng_bo)
        mu, sigma = gp.predict(cand, return_std=True)     # surrogate mean + uncertainty
        ei = expected_improvement(mu, sigma, best_f=y_obs.max())
        next_u = cand[int(np.argmax(ei))]                 # the most promising point to try next

        # evaluate the REAL (expensive) objective only at that one chosen point
        next_score = objective(decode(next_u))
        X_obs = np.vstack([X_obs, next_u])                # grow the observed dataset
        y_obs = np.append(y_obs, next_score)

    bayes_scores = y_obs
    bayes_best_params = decode(X_obs[int(np.argmax(y_obs))])
    bayes_backend = 'GP + Expected Improvement (fallback)'

# best-so-far efficiency curve, built exactly like the random-search one
bayes_best_curve = np.maximum.accumulate(bayes_scores)
print(f'\nBayesian ({bayes_backend}): best acc = {bayes_scores.max():.4f}')
print(f'Bayesian best params = {bayes_best_params}')

## 4. Efficiency comparison

We plot **best CV accuracy so far** against the **number of trials**. A curve that rises faster is more sample-efficient. Grid search has no natural trial ordering (its points are a fixed lattice), so we draw its final best score as a horizontal reference line and note its much larger fit budget.

In [ ]:
# ---- Efficiency curve: best-score-so-far vs number of trials ----
trials = np.arange(1, N_TRIALS + 1)

plt.figure(figsize=(8, 5))
plt.plot(trials, rand_best_curve, marker='o', label='Random search')
plt.plot(trials, bayes_best_curve, marker='s', label=f'Bayesian: {bayes_backend}')
# GridSearchCV has no meaningful trial ordering, so show its best score as a reference line.
plt.axhline(grid.best_score_, color='gray', ls='--',
            label=f'GridSearchCV best ({grid_fits} fits)')
plt.xlabel('number of trials (objective evaluations)')
plt.ylabel('best CV accuracy so far')
plt.title('Tuning efficiency: Bayesian vs Random vs Grid')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# ---- Tidy summary table of final results + budgets ----
summary = pd.DataFrame({
    'method':     ['GridSearchCV', 'Random search', f'Bayesian ({bayes_backend})'],
    'best_acc':   [grid.best_score_, rand_scores.max(), bayes_scores.max()],
    'model_fits': [grid_fits, N_TRIALS * CV.get_n_splits(), N_TRIALS * CV.get_n_splits()],
})
print(summary.to_string(index=False))

# how many trials Bayesian needed to first match/beat random search's FINAL best
_target = rand_best_curve[-1]
_reached = np.argmax(bayes_best_curve >= _target) + 1 if (bayes_best_curve >= _target).any() else None
if _reached:
    print(f'\nBayesian matched random-search final best ({_target:.4f}) after {_reached} trials '
          f'(random needed all {N_TRIALS}).')

## 5. Why BO beats grid/random — and the trade-offs

**Why it wins when evaluations are expensive.** Grid and random search are memoryless, so their expected best-so-far improves only slowly and their cost explodes with dimensionality. BO reuses every past trial through the surrogate, so it typically reaches a near-optimal score in far fewer evaluations — exactly what you want when one evaluation is a full CV fit (or training a deep net for hours).

**The acquisition trade-off.** Expected Improvement (and UCB, TPE, etc.) balance **exploitation** (sample where the mean is high) against **exploration** (sample where uncertainty is high). Too greedy and BO gets stuck in a local optimum; too exploratory and it wastes budget — the `xi` term tunes this balance.

**When BO is *not* worth it:**

- **Cheap objectives / huge budgets.** If a fit takes milliseconds, random search's simplicity and trivial parallelism win.
- **Sequential bottleneck.** Classic BO is inherently sequential (each trial depends on the last); grid/random parallelize perfectly across cores.
- **Surrogate overhead & high dimensions.** GPs scale as $O(n^3)$ in the number of trials and struggle in very high-dimensional or highly conditional spaces (where TPE / tree-based surrogates do better).
- **Noisy or non-stationary objectives** can mislead the surrogate.

**Rule of thumb:** the more expensive a single evaluation, the more Bayesian optimization pays off.